# Demographic Pyramids Across Data Sources (Figure 1)

Author: Bram Schonfeldt

This notebook generates Figure 1 of the manuscript: a composite 2x3 grid of overlapping horizontal population pyramids comparing the age and sex distributions of Fitbit commercial device users against general population benchmarks (UN World Population Prospects) across three distinct data sources:
1. Japan (November): 2015 vs. 2023 snapshots.
2. United Kingdom (50+): 2021–2023 aggregated older adult cohort.
3. 28 Target Countries (Pooled): 2015, 2020, and 2025 global longitudinal snapshots.

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import os
import pickle as pkl
import sys
from time import time
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Set rcParams
plt.rcdefaults()
plt.rcParams['figure.dpi'] = 200

sys.path.insert(0, os.path.abspath('../..'))
import stratify as strat

Load in the data...

In [ ]:
# Cohort configuration and base paths for internal and public datasets
start_date = '2015-01-01'
end_date = '2026-01-01'
n_shards = 100

internal_dir = strat.get_internal_data_dir()
public_dir = strat.get_public_data_dir()
un_pop_male_path = (
    f'{public_dir}/WPP2024_POP_F02_2_POPULATION_5-YEAR_AGE_GROUPS_MALE.xlsx'
)
un_pop_female_path = (
    f'{public_dir}/WPP2024_POP_F02_3_POPULATION_5-YEAR_AGE_GROUPS_FEMALE.xlsx'
)


# Shared helper functions for age midpoints
def midage_10(age: int) -> int:
  return age // 10 * 10 + 5 if age < 70 else 75


def midage_10_uk50(age: int) -> int:
  if age < 60:
    return 55
  elif age < 70:
    return 65
  elif age < 80:
    return 75
  else:
    return 85


# Load global and Japan step grids from internal_dir.
# Precomputed MD5 checkpoint hashes correspond to:
# - a36731bea99396ebe673f464d73ada56: Global 4D grid (target_country=None,
#   min_age=20, max_age=104, min_days_per_user=1, max_steps=24*60*60*2)
# - 9f4397029d8039e43903bb5b8b7e75ec: Japan-only 4D grid (target_country=
#   'Japan', start_month=11, stop_month=11, exclude_days_of_week=[6],
#   min_steps=100, max_steps=50000, otherwise same as above).
step_grid = strat.StratifySteps.from_shards(
    path_base=internal_dir,
    start_date=start_date,
    end_date=end_date,
    n_shards=n_shards,
    un_pop_male_path=un_pop_male_path,
    un_pop_female_path=un_pop_female_path,
    min_days_per_user=1,
    min_age=20,
    max_age=104,
    min_steps=None,
    max_steps=172800,
    target_country=None,
    start_month=None,
    stop_month=None,
    exclude_days_of_week=None,
    override_hash='a36731bea99396ebe673f464d73ada56',
)

step_grid_jp = strat.StratifySteps.from_shards(
    path_base=internal_dir,
    start_date=start_date,
    end_date=end_date,
    n_shards=n_shards,
    un_pop_male_path=un_pop_male_path,
    un_pop_female_path=un_pop_female_path,
    min_days_per_user=1,
    min_age=20,
    max_age=104,
    min_steps=100,
    max_steps=50000,
    target_country='Japan',
    start_month=11,
    stop_month=11,
    exclude_days_of_week=[6],
    override_hash='9f4397029d8039e43903bb5b8b7e75ec',
)

# Load WHO trends from public_dir to derive valid 28 target countries
who_trends_path = f'{public_dir}/who_trends.csv'
try:
  with strat.open_file(who_trends_path, 'r') as f:
    who_df = pd.read_csv(f)
except Exception:
  who_df = pd.read_csv('who_trends.csv')

who_countries = set(who_df['Country / Territory'].unique())

fb_df = step_grid.results_grid.reset_index()
fb_df['midage_10'] = fb_df['midage'].apply(midage_10)
fb_df['gender'] = fb_df['gender'].replace({0: 'male', 1: 'female'})

un_df = step_grid.un_pop.copy()
un_df['midage_10'] = un_df['age'].apply(midage_10)
un_df['gender'] = un_df['gender'].replace({0: 'male', 1: 'female'})

fb_subset = fb_df[(fb_df['year'] >= 2015) & (fb_df['year'] <= 2022)].copy()
fb_group_counts = (
    fb_subset.groupby(['country', 'year', 'gender', 'midage_10'])['readings']
    .sum()
    .reset_index()
)
low_count_mask = fb_group_counts['readings'] < 30
fb_group_counts['country_year'] = (
    fb_group_counts['country'].astype(str)
    + '_'
    + fb_group_counts['year'].astype(str)
)
low_count_keys = fb_group_counts[low_count_mask]['country_year'].unique()
fb_subset['country_year'] = (
    fb_subset['country'].astype(str) + '_' + fb_subset['year'].astype(str)
)
fb_filtered = fb_subset[~fb_subset['country_year'].isin(low_count_keys)].copy()
valid_years = (
    fb_filtered.groupby(['country', 'gender'])['year'].nunique().reset_index()
)

target_countries = []
for country in fb_filtered['country'].unique():
  m_yrs = valid_years[
      (valid_years['country'] == country) & (valid_years['gender'] == 'male')
  ]['year'].values
  f_yrs = valid_years[
      (valid_years['country'] == country) & (valid_years['gender'] == 'female')
  ]['year'].values
  if len(m_yrs) > 0 and len(f_yrs) > 0 and m_yrs[0] >= 5 and f_yrs[0] >= 5:
    if country in who_countries:
      target_countries.append(country)

target_countries = sorted(target_countries)

# 1. Japan (November)
jp_fb_df = step_grid_jp.results_grid.reset_index()
jp_fb_df['midage_10'] = jp_fb_df['midage'].apply(midage_10)
jp_fb_df['gender'] = jp_fb_df['gender'].replace({0: 'male', 1: 'female'})

jp_un = (
    un_df[un_df['country'] == 'Japan']
    .groupby(['year', 'gender', 'midage_10'])['Count']
    .sum()
    .reset_index()
    .rename(columns={'midage_10': 'age', 'Count': 'count'})
)
# UN WPP population counts are reported in thousands
jp_un['count'] *= 1000

jp_fb = (
    jp_fb_df[jp_fb_df['country'] == 'Japan']
    .groupby(['year', 'gender', 'midage_10'])['readings']
    .sum()
    .reset_index()
    .rename(columns={'midage_10': 'age', 'readings': 'count'})
)

# 2. UK 50+ Participants (Aggregated 2021-2023)
un_df_uk50 = un_df[
    (un_df['country'] == 'United Kingdom')
    & (un_df['age'] >= 50)
    & (un_df['year'] >= 2021)
    & (un_df['year'] <= 2023)
].copy()
un_df_uk50['midage_uk'] = un_df_uk50['age'].apply(midage_10_uk50)

un_by_year = (
    un_df_uk50.groupby(['year', 'gender', 'midage_uk'])['Count']
    .sum()
    .reset_index()
)
uk_un_agg_50plus = (
    un_by_year.groupby(['gender', 'midage_uk'])['Count']
    .mean()
    .reset_index()
    .rename(columns={'midage_uk': 'age', 'Count': 'count'})
)
# UN WPP population counts are reported in thousands
uk_un_agg_50plus['count'] *= 1000
uk_un_agg_50plus['year'] = 2022

# Load participant-level UK 50+ dataset checkpoint from internal_dir
output_uk_path_wave10 = (
    f'{internal_dir}/2021_2023_uk_50plus_participant_steps.pkl'
)
output_uk_path_legacy = (
    f'{internal_dir}/2021_2024_uk_50plus_participant_steps.pkl'
)

if strat.file_exists(output_uk_path_wave10):
  uk_participant_df = strat.load_pickle(output_uk_path_wave10)
  print(
      'Loaded UK participant-level Wave 10 dataset (June 2021 - March 2023)'
      f' with {len(uk_participant_df):,} records.'
  )
elif strat.file_exists(output_uk_path_legacy):
  uk_participant_df = strat.load_pickle(output_uk_path_legacy)
  print(
      'Loaded UK participant-level dataset with'
      f' {len(uk_participant_df):,} records.'
  )
else:
  checkpoint_dir = f'{internal_dir}/checkpoints_uk_50plus_elsa_wave10'
  if not strat.file_exists(checkpoint_dir):
    checkpoint_dir = f'{internal_dir}/checkpoints_uk_50plus'
  uk_dfs = []
  for modulo in range(n_shards):
    ckpt_file = f'{checkpoint_dir}/uk_50plus_shard_{modulo}.pkl'
    if strat.file_exists(ckpt_file):
      p_df = strat.load_pickle(ckpt_file)
      if p_df is not None and not p_df.empty:
        uk_dfs.append(p_df)
  if uk_dfs:
    uk_participant_df = pd.concat(uk_dfs, ignore_index=True)
    print(
        'Consolidated UK participant dataset from checkpoints:'
        f' {len(uk_participant_df):,} records.'
    )
  else:
    uk_participant_df = pd.DataFrame()
    print('Warning: UK participant dataset checkpoint not found.')

if not uk_participant_df.empty:
  uk_participant_df = uk_participant_df.copy()
  if (
      'midage_uk' not in uk_participant_df.columns
      and 'age' in uk_participant_df.columns
  ):
    uk_participant_df['midage_uk'] = uk_participant_df['age'].apply(
        midage_10_uk50
    )
  uk_participant_df['gender'] = (
      uk_participant_df['gender']
      .replace({0: 'male', 1: 'female'})
      .astype(str)
      .str.lower()
  )

  if 'user_id' in uk_participant_df.columns:
    uk_fb_agg_50plus = (
        uk_participant_df.groupby(['gender', 'midage_uk'])['user_id']
        .nunique()
        .reset_index(name='count')
        .rename(columns={'midage_uk': 'age'})
    )
  else:
    uk_fb_agg_50plus = (
        uk_participant_df.groupby(['gender', 'midage_uk'])
        .size()
        .reset_index(name='count')
        .rename(columns={'midage_uk': 'age'})
    )
  uk_fb_agg_50plus['year'] = 2022
else:
  fb_df_uk50 = fb_df[
      (fb_df['country'] == 'United Kingdom')
      & (fb_df['midage'] >= 50)
      & (fb_df['year'] >= 2021)
      & (fb_df['year'] <= 2023)
  ].copy()
  fb_df_uk50['midage_uk'] = fb_df_uk50['midage'].apply(midage_10_uk50)

  uk_fb_agg_50plus = (
      fb_df_uk50.groupby(['gender', 'midage_uk'])['readings']
      .sum()
      .reset_index()
      .rename(columns={'midage_uk': 'age', 'readings': 'count'})
  )
  uk_fb_agg_50plus['year'] = 2022

# 3. 28 Target Countries (Pooled)
c28_un = (
    un_df[un_df['country'].isin(target_countries)]
    .groupby(['year', 'gender', 'midage_10'])['Count']
    .sum()
    .reset_index()
    .rename(columns={'midage_10': 'age', 'Count': 'count'})
)
# UN WPP population counts are reported in thousands
c28_un['count'] *= 1000

c28_fb = (
    fb_df[fb_df['country'].isin(target_countries)]
    .groupby(['year', 'gender', 'midage_10'])['readings']
    .sum()
    .reset_index()
    .rename(columns={'midage_10': 'age', 'readings': 'count'})
)

pyramid_panels_data = [
    [
        ('Japan (November)', jp_un, jp_fb, 2015),
        ('Japan (November)', jp_un, jp_fb, 2023),
        ('United Kingdom (50+)', uk_un_agg_50plus, uk_fb_agg_50plus, 2022),
    ],
    [
        ('28 Target Countries (Pooled)', c28_un, c28_fb, 2015),
        ('28 Target Countries (Pooled)', c28_un, c28_fb, 2020),
        ('28 Target Countries (Pooled)', c28_un, c28_fb, 2025),
    ],
]

In [ ]:
def plot_composite_overlapping_pyramids(
    panels_data: List[List[Tuple[str, pd.DataFrame, pd.DataFrame, int]]],
    un_color: str = '#d3d3d3',
    un_male_color: Optional[str] = None,
    un_female_color: Optional[str] = None,
    fb_male_color: str = 'coral',
    fb_female_color: str = 'cornflowerblue',
    un_alpha: float = 0.5,
    fb_alpha: float = 0.65,
    bar_width: float = 8.0,
    wider_background_bars: bool = True,
    show_diff_annotations: bool = True,
) -> Tuple[plt.Figure, np.ndarray]:
  """Plots a 2x3 composite figure of overlapping horizontal population pyramids.

  Row 0: Japan 2015, Japan 2023, United Kingdom (50+) 2021-2023 Aggregated.
  Row 1: 28 Target Countries (Pooled) 2015, 2020, 2025.
  """
  num_rows = len(panels_data)
  num_cols = len(panels_data[0])
  fig, axes = plt.subplots(
      num_rows, num_cols, figsize=(7.5, 2.4 * num_rows), sharey=False
  )

  bg_height = bar_width * 0.88 if wider_background_bars else bar_width
  fg_height = bar_width * 0.65 if wider_background_bars else bar_width

  u_m_col = un_male_color if un_male_color else un_color
  u_f_col = un_female_color if un_female_color else un_color

  for r_idx in range(num_rows):
    for c_idx in range(num_cols):
      ax = axes[r_idx, c_idx]
      r_label, u_data, f_data, cur_year = panels_data[r_idx][c_idx]

      u_data = u_data.copy()
      u_data['gender'] = u_data['gender'].astype(str).str.lower()
      f_data = f_data.copy()
      f_data['gender'] = f_data['gender'].astype(str).str.lower()

      row_ages = sorted(
          list(set(u_data['age'].unique()).union(set(f_data['age'].unique())))
      )

      u_d = u_data[u_data['year'] == cur_year]
      f_d = f_data[f_data['year'] == cur_year]

      if not u_d.empty and not f_d.empty:
        u_tot = u_d['count'].sum() or 1
        f_tot = f_d['count'].sum() or 1

        u_m = (
            u_d[u_d['gender'] == 'male']
            .set_index('age')['count']
            .reindex(row_ages, fill_value=0)
            * 100
            / u_tot
        )
        u_f = (
            u_d[u_d['gender'] == 'female']
            .set_index('age')['count']
            .reindex(row_ages, fill_value=0)
            * 100
            / u_tot
        )
        f_m = (
            f_d[f_d['gender'] == 'male']
            .set_index('age')['count']
            .reindex(row_ages, fill_value=0)
            * 100
            / f_tot
        )
        f_f = (
            f_d[f_d['gender'] == 'female']
            .set_index('age')['count']
            .reindex(row_ages, fill_value=0)
            * 100
            / f_tot
        )

        bg_suffix = ' (wider bar)' if wider_background_bars else ''
        u_m_lbl = (
            f'General Pop - Male{bg_suffix}'
            if un_male_color
            else f'General Population{bg_suffix}'
        )
        u_f_lbl = (
            f'General Pop - Female{bg_suffix}' if un_female_color else None
        )

        # 1. Background General Population (No outline edge)
        ax.barh(
            row_ages,
            -u_m,
            color=u_m_col,
            alpha=un_alpha,
            edgecolor='none',
            height=bg_height,
            label=u_m_lbl,
            zorder=1,
        )
        ax.barh(
            row_ages,
            u_f,
            color=u_f_col,
            alpha=un_alpha,
            edgecolor='none',
            height=bg_height,
            label=u_f_lbl,
            zorder=1,
        )

        # 2. Foreground Fitbit Users
        ax.barh(
            row_ages,
            -f_m,
            color=fb_male_color,
            alpha=fb_alpha,
            edgecolor='none',
            height=fg_height,
            label='Male participants',
            zorder=2,
        )
        ax.barh(
            row_ages,
            f_f,
            color=fb_female_color,
            alpha=fb_alpha,
            edgecolor='none',
            height=fg_height,
            label='Female participants',
            zorder=2,
        )

        # Manually specified group X-limits and X-ticks:
        # - Group 1 (Japan 2015 & 2023, Row 0 Cols 0 & 1): xlim=(-30, 30), xticks=[-30, -15, 0, 15, 30]
        # - UK 50+ (Row 0 Col 2): xlim=(-40, 40), xticks=[-40, -20, 0, 20, 40]
        # - Group 2 (28 Target Countries Pooled, All Row 1): xlim=(-15, 15), xticks=[-15, -10, -5, 0, 5, 10, 15]
        if r_idx == 0 and c_idx < 2:
          panel_xlim = 40
          panel_ticks = [-40, -20, 0, 20, 40]
        elif r_idx == 0 and c_idx == 2:
          panel_xlim = 40
          panel_ticks = [-40, -20, 0, 20, 40]
        else:
          panel_xlim = 20
          panel_ticks = [-20, -10, 0, 10, 20]

        # 3. Annotations (+/- zz without % sign) for over/under representation per bar
        if show_diff_annotations:
          offset = panel_xlim * 0.015
          for age_val in row_ages:
            u_m_v = u_m.loc[age_val] if age_val in u_m.index else 0.0
            f_m_v = f_m.loc[age_val] if age_val in f_m.index else 0.0
            u_f_v = u_f.loc[age_val] if age_val in u_f.index else 0.0
            f_f_v = f_f.loc[age_val] if age_val in f_f.index else 0.0

            diff_m = f_m_v - u_m_v
            diff_f = f_f_v - u_f_v

            x_m = -max(u_m_v, f_m_v) - offset
            x_f = max(u_f_v, f_f_v) + offset

            text_m = f'{diff_m:+.1f}'
            text_f = f'{diff_f:+.1f}'

            ax.text(
                x_m,
                age_val,
                text_m,
                ha='right',
                va='center',
                fontsize=8,
                color='#333333',
                zorder=3,
            )
            ax.text(
                x_f,
                age_val,
                text_f,
                ha='left',
                va='center',
                fontsize=8,
                color='#333333',
                zorder=3,
            )

        ax.set_xlim(-panel_xlim * 1.10, panel_xlim * 1.10)
        ax.set_xticks(panel_ticks)
        ax.set_xticklabels([f'{abs(x)}' for x in panel_ticks])

        year_str = '2021-2023' if r_label == 'United Kingdom' else f'{cur_year}'
        n_str = f'Population n={int(u_tot/1e6):,}M\nFitbit n={int(f_tot):,}'

        if r_idx == 1 and c_idx != 1:
          ax.set_title(f'{year_str}\n{n_str}')
        else:
          ax.set_title(f'{r_label}\n{year_str}\n{n_str}')

        if c_idx == 0:
          ax.set_ylabel('Age Group', rotation=0, ha='right')

        if c_idx == 0 or r_label == 'United Kingdom':
          ax.set_yticks(row_ages)
          if r_label == 'United Kingdom':
            y_labels = [
                '50-59'
                if a == 55
                else '60-69'
                if a == 65
                else '70-79'
                if a == 75
                else '80+'
                for a in row_ages
            ]
          else:
            y_labels = [
                f'{int(a-5)}-{int(a+4)}' if a < 75 else '70+' for a in row_ages
            ]
          ax.set_yticklabels(y_labels)
        else:
          ax.set_yticks(row_ages)
          ax.set_yticklabels([])
          ax.tick_params(axis='y', which='both', length=0)

        if r_idx == num_rows - 1:
          ax.set_xlabel('Percentage (%)')

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        if c_idx != 0 and r_label != 'United Kingdom':
          ax.spines['left'].set_visible(False)
      else:
        ax.text(0.5, 0.5, 'No Data', ha='center', va='center')

  plt.tight_layout()
  return fig, axes


pyramid_panels_data = [
    [
        ('Japan', jp_un, jp_fb, 2015),
        ('Japan', jp_un, jp_fb, 2023),
        ('United Kingdom', uk_un_agg_50plus, uk_fb_agg_50plus, 2022),
    ],
    [
        ('Countries with large Fitbit user bases', c28_un, c28_fb, 2015),
        ('Countries with large Fitbit user bases', c28_un, c28_fb, 2020),
        ('Countries with large Fitbit user bases', c28_un, c28_fb, 2025),
    ],
]

# Render composite 2x3 population pyramids figure (Figure 1)
fig, axes = plot_composite_overlapping_pyramids(
    panels_data=pyramid_panels_data,
    un_color='#d3d3d3',
    un_male_color=None,
    un_female_color=None,
    fb_male_color='coral',
    fb_female_color='cornflowerblue',
    un_alpha=0.5,
    fb_alpha=0.65,
    wider_background_bars=True,
    show_diff_annotations=True,
)

handles, labels = axes[0, 0].get_legend_handles_labels()
unique_labels = dict(zip(labels, handles))
fig.suptitle(
    'Demographic distribution of Fitbit users compared to the general'
    ' population',
    y=0.98,
)
fig.legend(
    list(unique_labels.values()),
    list(unique_labels.keys()),
    loc='lower center',
    ncol=len(unique_labels),
    bbox_to_anchor=(0.55, -0.05),
    handlelength=2.2,
    handleheight=1.2,
)
fig.tight_layout()
plt.show()

# *FIN*